# ResNet50 Classification Modules

**Purpose:** Shared modules for all ResNet50 classification experiments.

**Usage:** Run this notebook first, then use `%run ./ResNet50_modules.ipynb` in experiment notebooks.

## Cell 1: Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, models, datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import json, os, copy

%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Cell 2: ResNet50Classifier Model

In [ ]:
class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.5, pretrained=True,
                 fc_hidden_dims=None, use_batch_norm=True, modify_backbone=False,
                 remove_layer=None, add_conv_after_layer=None):
        super(ResNet50Classifier, self).__init__()
        
        if remove_layer and remove_layer not in ['layer3', 'layer4']:
            raise ValueError("remove_layer must be 'layer3' or 'layer4'")
        if add_conv_after_layer and add_conv_after_layer not in ['layer1', 'layer2', 'layer3']:
            raise ValueError("add_conv_after_layer must be 'layer1', 'layer2', or 'layer3'")
        if remove_layer and add_conv_after_layer:
            raise ValueError("Cannot both remove and add layers")
        
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base_model = models.resnet50(weights=weights)
        
        if modify_backbone:
            self.backbone = self._modify_backbone(base_model, remove_layer, add_conv_after_layer)
        else:
            self.backbone = base_model
        
        num_features = self._get_feature_dimension()
        self.backbone.fc = nn.Identity()
        
        if fc_hidden_dims is None or len(fc_hidden_dims) == 0:
            self.classifier = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(num_features, num_classes))
        else:
            layers = []
            current_dim = num_features
            for hidden_dim in fc_hidden_dims:
                layers.extend([nn.Dropout(dropout_rate), nn.Linear(current_dim, hidden_dim)])
                if use_batch_norm:
                    layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.ReLU())
                current_dim = hidden_dim
            layers.extend([nn.Dropout(dropout_rate), nn.Linear(current_dim, num_classes)])
            self.classifier = nn.Sequential(*layers)
    
    def _get_feature_dimension(self):
        dummy_input = torch.zeros(1, 3, 224, 224)
        training_mode = self.backbone.training
        self.backbone.eval()
        with torch.no_grad():
            x = self.backbone.conv1(dummy_input)
            x = self.backbone.bn1(x)
            x = self.backbone.relu(x)
            x = self.backbone.maxpool(x)
            x = self.backbone.layer1(x)
            x = self.backbone.layer2(x)
            x = self.backbone.layer3(x)
            x = self.backbone.layer4(x)
            x = self.backbone.avgpool(x)
            x = torch.flatten(x, 1)
            num_features = x.shape[1]
        self.backbone.train(training_mode)
        return num_features
    
    def _modify_backbone(self, base_model, remove_layer=None, add_conv_after_layer=None):
        modified = copy.deepcopy(base_model)
        
        if remove_layer == 'layer3':
            modified.layer3 = nn.Identity()
            original_conv1 = modified.layer4[0].conv1
            modified.layer4[0].conv1 = nn.Conv2d(512, original_conv1.out_channels, original_conv1.kernel_size, original_conv1.stride, original_conv1.padding, bias=original_conv1.bias is not None)
            if modified.layer4[0].downsample is not None:
                orig_ds = modified.layer4[0].downsample[0]
                modified.layer4[0].downsample[0] = nn.Conv2d(512, orig_ds.out_channels, orig_ds.kernel_size, orig_ds.stride, orig_ds.padding, bias=orig_ds.bias is not None)
            print("✓ Removed layer3")
        elif remove_layer == 'layer4':
            modified.layer4 = nn.Identity()
            print("✓ Removed layer4")
        
        if add_conv_after_layer:
            num_channels = self._get_original_layer_channels(modified, add_conv_after_layer)
            conv_block = nn.Sequential(
                nn.Conv2d(num_channels, num_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(num_channels), nn.ReLU(inplace=True),
                nn.Conv2d(num_channels, num_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(num_channels), nn.ReLU(inplace=True)
            )
            if add_conv_after_layer == 'layer1':
                modified.layer1 = nn.Sequential(modified.layer1, conv_block)
                print("✓ Added conv block after layer1")
            elif add_conv_after_layer == 'layer2':
                modified.layer2 = nn.Sequential(modified.layer2, conv_block)
                print("✓ Added conv block after layer2")
            elif add_conv_after_layer == 'layer3':
                modified.layer3 = nn.Sequential(modified.layer3, conv_block)
                print("✓ Added conv block after layer3")
        return modified
    
    def _get_original_layer_channels(self, model, layer_name):
        if layer_name == 'layer1': return model.layer1[-1].conv3.out_channels
        elif layer_name == 'layer2': return model.layer2[-1].conv3.out_channels
        elif layer_name == 'layer3': return model.layer3[-1].conv3.out_channels
        elif layer_name == 'layer4': return model.layer4[-1].conv3.out_channels
        return 512

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

print("✓ ResNet50Classifier defined")

## Cell 3: Training Framework (TrainingConfig Dataclass)

In [ ]:
@dataclass
class TrainingConfig:
    learning_rate: float = 5e-4
    weight_decay: float = 5e-4
    optimizer_type: str = 'adamw'
    epochs: int = 200
    use_scheduler: bool = True
    scheduler_type: str = 'reduce_on_plateau'
    scheduler_patience: int = 7
    scheduler_factor: float = 0.5
    use_warmup: bool = True
    warmup_epochs: int = 10
    use_early_stopping: bool = True
    early_stopping_patience: int = 50
    label_smoothing: float = 0.1
    use_amp: bool = True
    description: str = 'Default training configuration'

print("✓ TrainingConfig dataclass defined")

## Cell 4: ClassificationTrainer

In [ ]:
class ClassificationTrainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        
        trainable_params = filter(lambda p: p.requires_grad, model.parameters())
        if config.optimizer_type == 'adamw':
            self.optimizer = torch.optim.AdamW(trainable_params, lr=config.learning_rate, weight_decay=config.weight_decay)
        elif config.optimizer_type == 'adam':
            self.optimizer = torch.optim.Adam(trainable_params, lr=config.learning_rate, weight_decay=config.weight_decay)
        else:
            self.optimizer = torch.optim.SGD(trainable_params, lr=config.learning_rate, momentum=0.9, weight_decay=config.weight_decay)
        
        # Use new API to avoid deprecation warning
        try:
            self.scaler = torch.amp.GradScaler('cuda') if config.use_amp else None
        except:
            self.scaler = torch.cuda.amp.GradScaler() if config.use_amp else None
        
        self.scheduler = None
        if config.use_scheduler:
            self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience)
        
        self.best_val_acc = 0.0
        self.training_history = []
    
    def print_model_summary(self):
        """Print detailed model architecture summary."""
        print("\n" + "="*80)
        print("MODEL ARCHITECTURE SUMMARY")
        print("="*80)
        
        batch_size = 2
        dummy_input = torch.randn(batch_size, 3, 224, 224).to(self.device)
        
        original_training = self.model.training
        self.model.eval()
        
        layer_info = []
        total_params = 0
        trainable_params = 0
        
        def register_hook(module):
            def hook(module, input, output):
                class_name = str(module.__class__.__name__)
                if 'Sequential' in class_name or 'ModuleList' in class_name:
                    return
                
                module_name = module._get_name() if hasattr(module, '_get_name') else class_name
                module_params = sum(p.numel() for p in module.parameters())
                module_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
                
                input_shape = list(input[0].size()) if isinstance(input, tuple) else list(input.size())
                output_shape = list(output[0].size()) if isinstance(output, tuple) else list(output.size())
                
                layer_info.append({
                    'name': module_name,
                    'input_shape': input_shape,
                    'output_shape': output_shape,
                    'params': module_params,
                    'trainable': module_trainable
                })
            
            if not list(module.children()):
                module.register_forward_hook(hook)
        
        self.model.apply(register_hook)
        
        with torch.no_grad():
            _ = self.model(dummy_input)
        
        if original_training:
            self.model.train()
        
        print(f"\n{'Layer Name':<30} {'Input Shape':<20} {'Output Shape':<20} {'Params':<12} {'Trainable':<12}")
        print("-"*80)
        
        for info in layer_info:
            input_str = f"[{', '.join(map(str, info['input_shape']))}]"
            output_str = f"[{', '.join(map(str, info['output_shape']))}]"
            params_str = f"{info['params']:,}"
            trainable_str = f"{info['trainable']:,}"
            
            print(f"{info['name']:<30} {input_str:<20} {output_str:<20} {params_str:<12} {trainable_str:<12}")
            
            total_params += info['params']
            trainable_params += info['trainable']
        
        print("-"*80)
        print(f"\nTotal Parameters: {total_params:,}")
        print(f"Trainable Parameters: {trainable_params:,}")
        print(f"Non-trainable Parameters: {total_params - trainable_params:,}")
        print(f"Model Size (approx): {total_params * 4 / (1024**2):.2f} MB")
        print("="*80 + "\n")
    
    def train_epoch(self, train_loader, criterion, epoch):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        
        for inputs, targets in progress_bar:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            
            if self.config.use_amp:
                # Use new API to avoid deprecation warning
                try:
                    autocast = torch.amp.autocast('cuda')
                except:
                    autocast = torch.cuda.amp.autocast()
                
                with autocast:
                    outputs = self.model(inputs)
                    loss = criterion(outputs, targets)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
            else:
                outputs = self.model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
        
        return running_loss / len(train_loader), correct / total
    
    def validate(self, val_loader, criterion):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                if self.config.use_amp:
                    # Use new API to avoid deprecation warning
                    try:
                        autocast = torch.amp.autocast('cuda')
                    except:
                        autocast = torch.cuda.amp.autocast()
                    
                    with autocast:
                        outputs = self.model(inputs)
                        loss = criterion(outputs, targets)
                else:
                    outputs = self.model(inputs)
                    loss = criterion(outputs, targets)
                
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
        
        return running_loss / len(val_loader), correct / total
    
    def train(self, train_loader, val_loader, criterion, output_dir):
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        early_stop_counter = 0
        
        print(f'\nTraining: {self.config.description}')
        print(f'  Epochs: {self.config.epochs}, LR: {self.config.learning_rate}, Weight Decay: {self.config.weight_decay}')
        print(f'  Early Stopping: {"Enabled" if self.config.use_early_stopping else "Disabled"} (patience={self.config.early_stopping_patience})\n')
        
        for epoch in range(self.config.epochs):
            if self.config.use_warmup and epoch < self.config.warmup_epochs:
                warmup_ratio = (epoch + 1) / self.config.warmup_epochs
                current_lr = self.config.learning_rate * warmup_ratio
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = current_lr
            
            train_loss, train_acc = self.train_epoch(train_loader, criterion, epoch)
            val_loss, val_acc = self.validate(val_loader, criterion)
            
            if self.scheduler:
                self.scheduler.step(val_loss)
            
            current_lr = self.optimizer.param_groups[0]['lr']
            history_entry = {'epoch': epoch + 1, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc, 'lr': current_lr}
            self.training_history.append(history_entry)
            
            print(f'Epoch {epoch+1}/{self.config.epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
            
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                early_stop_counter = 0
                checkpoint_path = output_path / 'best_model.pth'
                torch.save({'model_state_dict': self.model.state_dict(), 'optimizer_state_dict': self.optimizer.state_dict(), 'val_acc': val_acc, 'epoch': epoch, 'config': self.config}, checkpoint_path)
                print(f'  ✓ Best model saved (Val Acc: {val_acc:.4f})')
            else:
                early_stop_counter += 1
            
            if self.config.use_early_stopping and early_stop_counter >= self.config.early_stopping_patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
        
        return {'history': self.training_history, 'best_val_acc': self.best_val_acc}

print("✓ ClassificationTrainer defined")

## Cell 5: Data Loading Utilities

In [ ]:
def create_classification_dataloaders(data_root, batch_size=16, num_workers=2, augmentation_type='none'):
    if augmentation_type == 'none':
        train_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    elif augmentation_type == 'standard':
        train_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    elif augmentation_type == 'enhanced':
        train_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.RandomHorizontalFlip(), transforms.RandomRotation(20), transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1), transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    else:
        raise ValueError(f"Unknown augmentation type: {augmentation_type}")
    
    test_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    
    train_dataset = datasets.ImageFolder(f'{data_root}/train', transform=train_transform)
    val_dataset = datasets.ImageFolder(f'{data_root}/valid', transform=test_transform)
    test_dataset = datasets.ImageFolder(f'{data_root}/test', transform=test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return train_loader, val_loader, test_loader, train_dataset.classes

print("✓ Data loading utilities defined")

## Cell 6: ClassificationEvaluator

In [ ]:
class ClassificationEvaluator:
    def __init__(self, class_names):
        self.class_names = class_names
        self.metrics = {}
    
    def evaluate(self, model, test_loader):
        print("=" * 80)
        print("MODEL EVALUATION")
        print("=" * 80)
        
        y_true, y_pred = self._get_predictions(model, test_loader)
        
        accuracy = accuracy_score(y_true, y_pred)
        precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        cm = confusion_matrix(y_true, y_pred)
        
        per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
        per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
        per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
        
        self.metrics = {
            'accuracy': float(accuracy), 'precision_weighted': float(precision_weighted),
            'recall_weighted': float(recall_weighted), 'f1_weighted': float(f1_weighted),
            'precision_macro': float(precision_macro), 'recall_macro': float(recall_macro),
            'f1_macro': float(f1_macro), 'per_class_precision': per_class_precision.tolist(),
            'per_class_recall': per_class_recall.tolist(), 'per_class_f1': per_class_f1.tolist(),
            'confusion_matrix': cm
        }
        
        print(f'\nOverall Metrics:')
        print(f'  Accuracy: {accuracy:.4f}')
        print(f'  Precision (weighted): {precision_weighted:.4f}')
        print(f'  Recall (weighted): {recall_weighted:.4f}')
        print(f'  F1-Score (weighted): {f1_weighted:.4f}')
        print(f'  Precision (macro): {precision_macro:.4f}')
        print(f'  Recall (macro): {recall_macro:.4f}')
        print(f'  F1-Score (macro): {f1_macro:.4f}')
        
        print(f'\nPer-Class Metrics:')
        for i, class_name in enumerate(self.class_names):
            print(f'  {class_name}: P={per_class_precision[i]:.4f} R={per_class_recall[i]:.4f} F1={per_class_f1[i]:.4f}')
        
        print(f'\nConfusion Matrix:\n{cm}')
        return self.metrics
    
    def analyze_overfitting(self, history):
        if not history or len(history) < 5:
            return {'pattern': 'insufficient_data', 'description': 'Not enough epochs', 'recommendation': 'Train for more epochs', 'gap': 0.0}
        
        recent_epochs = history[-5:]
        avg_train_acc = np.mean([h['train_acc'] for h in recent_epochs])
        avg_val_acc = np.mean([h['val_acc'] for h in recent_epochs])
        acc_gap = avg_train_acc - avg_val_acc
        
        val_losses_trend = [h['val_loss'] for h in history[-10:]]
        is_val_loss_increasing = len(val_losses_trend) > 1 and val_losses_trend[-1] > val_losses_trend[0]
        
        analysis = {'avg_train_acc': float(avg_train_acc), 'avg_val_acc': float(avg_val_acc), 'gap': float(acc_gap)}
        
        if acc_gap > 0.15 or is_val_loss_increasing:
            analysis['pattern'] = 'overfitting'
            analysis['description'] = f'Model shows overfitting. Train acc ({avg_train_acc:.4f}) >> Val acc ({avg_val_acc:.4f}).'
            analysis['recommendation'] = 'Consider: 1) Add dropout, 2) Increase weight decay, 3) Add data augmentation'
        elif avg_train_acc < 0.6 and avg_val_acc < 0.6:
            analysis['pattern'] = 'underfitting'
            analysis['description'] = f'Model shows underfitting. Both accuracies low.'
            analysis['recommendation'] = 'Consider: 1) Train longer, 2) Reduce regularization, 3) Increase capacity'
        else:
            analysis['pattern'] = 'good_fit'
            analysis['description'] = f'Model has good fit. Train acc ({avg_train_acc:.4f}) ≈ Val acc ({avg_val_acc:.4f}).'
            analysis['recommendation'] = 'Model performance is acceptable.'
        
        return analysis
    
    def _get_predictions(self, model, test_loader):
        model.eval()
        device = next(model.parameters()).device
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for inputs, targets in tqdm(test_loader, desc="Evaluating"):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                if isinstance(outputs, tuple):
                    main_out = outputs[0]
                else:
                    main_out = outputs
                _, predicted = main_out.max(1)
                all_preds.append(predicted.cpu().numpy())
                all_labels.append(targets.cpu().numpy())
        
        return np.concatenate(all_labels), np.concatenate(all_preds)

print("✓ ClassificationEvaluator defined")
print("\n✓ All modules loaded successfully!")